# Week 3: Exercise 6 - Filesystem Tools

**Goal:** Build the 5 filesystem tools that every agent needs.

No API key needed for this exercise.


## Step 1: Implement FileSystemTools

**TODO:** Implement all 5 methods.
Use `pathlib.Path`. Each method returns a string. Wrap in try/except.


In [4]:
from pathlib import Path

class FileSystemTools:
    """TODO: Implement all 5 methods."""

    def write_file(self, path, content, append=False):
        """TODO: Write content to a file. If append=True, add to end."""
        # Hint: use mode "a" for append, "w" for write
        #       use p.parent.mkdir(parents=True, exist_ok=True) to create dirs
        p = Path(path)
        p.parent.mkdir(parents = True, exist_ok = True)
        mode = "a" if append else "w"
        with open(p,mode,encoding = "utf-8") as op:
            op.write(content)

        return f"Write file {path} with {len(content.split('/'))}"

    def read_file(self, path, encoding="utf-8", limit=1000):
        """TODO: Read file contents, up to `limit` lines."""
        # Hint: Path(path).read_text(...).splitlines()[:limit]
        try:
            p = Path(path)
            lines = p.read_text(encoding = encoding).splitlines()[:limit]
            return "\n".join(lines)
        except Exception as E:
            print(f"Error: {E}")
            return None

    def list_files(self, path=".", pattern="*"):
        """TODO: List files matching a glob pattern."""
        # Hint: Path(path).glob(pattern)
        p = Path(path)
        matches = sorted(p.glob(pattern), key = lambda e : len(str(e)))
        return "\n".join(str(m) for m in matches)

    def search_in_files(self, query, path=".", file_pattern="*.", max_results=50):
        """TODO: Search for text inside files."""
        # Hint: Path(path).rglob(file_pattern), then read each file line by line
        results = []
        p = Path(path)
        matches = sorted(p.rglob(file_pattern), key = lambda e : len(str(e)))

        for file in matches:
            fpath = Path(file)

            lines = fpath.read_text(encoding = "utf-8").splitlines()
            for id, line in enumerate(lines,1):
                if query.lower() in line.lower():
                    results.append(f"{id} - {file} : {line}")
                if len(results) > max_results:
                    return "\n".join(results)

        return "\n".join(results)


    def delete_file(self, path):
        """TODO: Delete a file."""
        # Hint: Path(path).unlink()
        p = Path(path)
        p.unlink(missing_ok = True)
        
        return f"Delete {path} successfully!"
        


## Step 2: Implement get_tool_schemas
**TODO:** Return JSON schemas for all 5 tools.


In [5]:
def get_tool_schemas():
    """TODO: Return JSON schemas for all 5 tools."""
    # Hint: return a list of 5 {"type": "function", "function": {...}} dicts —
    #       write_file, read_file, list_files, search_in_files, delete_file
    return [
        {
            "type": "function",
            "function": {
                "name": "write_file",
                "description": "Write file!",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string", "description": "File's path(e.g: 'text.txt')."},
                        "content": {"type": "string", "description": "Content to be written into file."}
                    },
                    "required": ["path","content"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "read_file",
                "description": "Read file!",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string", "description": "File's path(e.g: 'text.txt')."},
                        "encoding": {"type": "string", "description": "Encoding file content."},
                        "limit": {"type": "integer", "description": "Limited number of lines."},
                    },
                    "required": ["path","limit"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "list_files",
                "description": "Return list of  files!",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string", "description": "File's path(e.g: 'text.txt')."},
                        "pattern": {"type": "string", "description": "File pattern(e.g: '.txt')."},
                    },
                    "required": ["path","pattern"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "search_in_files",
                "description": "Search query in files!",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "Query word/lines to find."},
                        "path": {"type": "string", "description": "File's path(e.g: 'text.txt')."},
                        "file_pattern": {"type": "string", "description": "File pattern(e.g: '.txt')."},
                        "max_results": {"type": "integer", "description": "Maximum number of results."}
                    },
                    "required": ["query","path","file_pattern"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "delete_file",
                "description": "Delete file!",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "path": {"type": "string", "description": "File's path(e.g: 'text.txt')."}
                    },
                    "required": ["path"]
                }
            }
        }
    ]


## Step 3: Test the Tools


In [6]:
fs = FileSystemTools()

print(fs.write_file("test.txt", "Hello World!\nLine 2 here."))
print(fs.read_file("test.txt"))
print(fs.list_files("."))
print(fs.search_in_files("Hello", "."))
print(fs.delete_file("test.txt"))


Write file test.txt with 1
Hello World!
Line 2 here.
test.txt
README.md
functions.md
exercise_7_tool_registry.ipynb
exercise_6_filesystem_tools.ipynb

Delete test.txt successfully!


## Key Takeaways
- Every tool needs: implementation + schema + error handling
- pathlib is cross-platform


In [7]:
# Test 6: get_tool_schemas returns valid schemas
print("--- Test 6: Tool schemas ---")

schemas = get_tool_schemas()
assert isinstance(schemas, list), "Should return a list"
assert len(schemas) == 5, f"Should have 5 tools, got {len(schemas)}"

# Check structure of each schema
names = []
for s in schemas:
    assert s["type"] == "function", f"type should be 'function', got {s.get('type')}"
    assert "function" in s, "Missing 'function' key"
    assert "name" in s["function"], "Missing function.name"
    assert "description" in s["function"], "Missing function.description"
    assert "parameters" in s["function"], "Missing function.parameters"
    names.append(s["function"]["name"])

print(f"  Tools found: {names}")
assert "write_file" in names, "Missing write_file schema"
assert "read_file" in names, "Missing read_file schema"
assert "list_files" in names, "Missing list_files schema"
assert "search_in_files" in names, "Missing search_in_files schema"
assert "delete_file" in names, "Missing delete_file schema"
print("Test 6 passed: All 5 schemas valid.")


--- Test 6: Tool schemas ---
  Tools found: ['write_file', 'read_file', 'list_files', 'search_in_files', 'delete_file']
Test 6 passed: All 5 schemas valid.
